In [ ]:
!nvidia-smi

Mon May  4 21:42:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#Si no esta instalado
!apt-get update
!apt-get install -y nvidia-cuda-toolkit

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%writefile hello.cu
#include <stdio.h>

__global__ void helloCUDA(float e){
    printf("Hola, Este es el hilo %d del bloque %d con un valor de e=%f\n", threadIdx.x, blockIdx.x, e);
}

int main(){
    helloCUDA<<<3, 4>>>(2.71828f);
    cudaDeviceSynchronize();
    cudaDeviceReset();
    return 0;
}

Overwriting hello.cu


In [ ]:
!nvcc hello.cu -o hello
!./hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hola, Este es el hilo 0 del bloque 2 con un valor de e=2.718280
Hola, Este es el hilo 1 del bloque 2 con un valor de e=2.718280
Hola, Este es el hilo 2 del bloque 2 con un valor de e=2.718280
Hola, Este es el hilo 3 del bloque 2 con un valor de e=2.718280
Hola, Este es el hilo 0 del bloque 0 con un valor de e=2.718280
Hola, Este es el hilo 1 del bloque 0 con un valor de e=2.718280
Hola, Este es el hilo 2 del bloque 0 con un valor de e=2.718280
Hola, Este es el hilo 3 del bloque 0 con un valor de e=2.718280
Hola, Este es el hilo 0 del bloque 1 con un valor de e=2.718280
Hola, Este es el hilo 1 del bloque 1 con un valor de e=2.718280
Hola, Este es el hilo 2 del bloque 1 con un valor de e=2.718280
Hola, Este es el hilo 3 del bloque 1 con un valor de e=2.718280


In [ ]:
%%writefile threads.cu
#include <stdio.h>

#define TAM 128

__global__ void idThreads_kernel(int *block_dev, int *threadLocal_dev, int *warp_dev, int *threadGlobal_dev){
    int threadGlobal_idx = (blockIdx.x * blockDim.x) + threadIdx.x;

    block_dev[threadGlobal_idx] = blockIdx.x;
    threadLocal_dev[threadGlobal_idx] = threadIdx.x;
    warp_dev[threadGlobal_idx] = threadGlobal_idx / warpSize;
    threadGlobal_dev[threadGlobal_idx] = threadGlobal_idx;
}

int main(void){
    int num_blocks = 4;
    int num_threads = 32;

    int block_host[TAM], threadLocal_host[TAM], warp_host[TAM], threadGlobal_host[TAM];

    int *block_dev, *threadLocal_dev, *warp_dev, *threadGlobal_dev;

    size_t TAM_Bytes_int = TAM * sizeof(int);

    cudaMalloc((void**)&block_dev, TAM_Bytes_int);
    cudaMalloc((void**)&threadLocal_dev, TAM_Bytes_int);
    cudaMalloc((void**)&warp_dev, TAM_Bytes_int);
    cudaMalloc((void**)&threadGlobal_dev, TAM_Bytes_int);

    idThreads_kernel<<<num_blocks, num_threads>>>(block_dev, threadLocal_dev, warp_dev, threadGlobal_dev);

    cudaMemcpy(block_host, block_dev, TAM_Bytes_int, cudaMemcpyDeviceToHost);
    cudaMemcpy(threadLocal_host, threadLocal_dev, TAM_Bytes_int, cudaMemcpyDeviceToHost);
    cudaMemcpy(warp_host, warp_dev, TAM_Bytes_int, cudaMemcpyDeviceToHost);
    cudaMemcpy(threadGlobal_host, threadGlobal_dev, TAM_Bytes_int, cudaMemcpyDeviceToHost);

    for(int i=0;i<10;i++){
        printf("Global %d | Block %d | Local %d | Warp %d\n",
            threadGlobal_host[i], block_host[i], threadLocal_host[i], warp_host[i]);
    }

    cudaFree(block_dev);
    cudaFree(threadLocal_dev);
    cudaFree(warp_dev);
    cudaFree(threadGlobal_dev);

    return 0;
}

Writing threads.cu


In [ ]:
!nvcc threads.cu -o threads
!./threads

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Global 0 | Block 0 | Local 0 | Warp 0
Global 1 | Block 0 | Local 1 | Warp 0
Global 2 | Block 0 | Local 2 | Warp 0
Global 3 | Block 0 | Local 3 | Warp 0
Global 4 | Block 0 | Local 4 | Warp 0
Global 5 | Block 0 | Local 5 | Warp 0
Global 6 | Block 0 | Local 6 | Warp 0
Global 7 | Block 0 | Local 7 | Warp 0
Global 8 | Block 0 | Local 8 | Warp 0
Global 9 | Block 0 | Local 9 | Warp 0


In [ ]:
%%writefile suma_vectores.cu
#include <stdio.h>

__global__ void Suma_vectores(float *c_d, float *a_d, float *b_d, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        c_d[idx] = a_d[idx] + b_d[idx];
    }
}

int main(void){
    int N = 24;
    size_t size = N * sizeof(float);

    float a_h[N], b_h[N], c_h[N];

    for(int i=0;i<N;i++){
        a_h[i] = i;
        b_h[i] = 2*i;
    }

    float *a_d, *b_d, *c_d;

    cudaMalloc((void**)&a_d, size);
    cudaMalloc((void**)&b_d, size);
    cudaMalloc((void**)&c_d, size);

    cudaMemcpy(a_d, a_h, size, cudaMemcpyHostToDevice);
    cudaMemcpy(b_d, b_h, size, cudaMemcpyHostToDevice);

    int block_size = 8;
    int n_blocks = (N + block_size - 1)/block_size;

    Suma_vectores<<<n_blocks, block_size>>>(c_d, a_d, b_d, N);

    cudaMemcpy(c_h, c_d, size, cudaMemcpyDeviceToHost);

    for(int i=0;i<N;i++){
        printf("%f + %f = %f\n", a_h[i], b_h[i], c_h[i]);
    }

    cudaFree(a_d);
    cudaFree(b_d);
    cudaFree(c_d);

    return 0;
}

Writing suma_vectores.cu


In [ ]:
!nvcc suma_vectores.cu -o suma
!./suma

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
0.000000 + 0.000000 = 0.000000
1.000000 + 2.000000 = 3.000000
2.000000 + 4.000000 = 6.000000
3.000000 + 6.000000 = 9.000000
4.000000 + 8.000000 = 12.000000
5.000000 + 10.000000 = 15.000000
6.000000 + 12.000000 = 18.000000
7.000000 + 14.000000 = 21.000000
8.000000 + 16.000000 = 24.000000
9.000000 + 18.000000 = 27.000000
10.000000 + 20.000000 = 30.000000
11.000000 + 22.000000 = 33.000000
12.000000 + 24.000000 = 36.000000
13.000000 + 26.000000 = 39.000000
14.000000 + 28.000000 = 42.000000
15.000000 + 30.000000 = 45.000000
16.000000 + 32.000000 = 48.000000
17.000000 + 34.000000 = 51.000000
18.000000 + 36.000000 = 54.000000
19.000000 + 38.000000 = 57.000000
20.000000 + 40.000000 = 60.000000
21.000000 + 42.000000 = 63.000000
22.000000 + 44.000000 = 66.000000
23.000000 + 46.000000 = 69.000000


In [ ]:
%%writefile matmul.cu
#include <stdio.h>

#define BLOCK_SIZE 16

__global__ void Multiplica_Matrices_GM(float *C, float *A, float *B, int nfil, int ncol) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int idy = blockIdx.y * blockDim.y + threadIdx.y;

    if (idy < nfil && idx < ncol) {
        float sum = 0.0f;
        for (int k = 0; k < ncol; k++) {
            sum += A[idy * ncol + k] * B[k * ncol + idx];
        }
        C[idy * ncol + idx] = sum;
    }
}

int div_up(int a, int b){
    return (a + b - 1)/b;
}

int main(){
    int n = 4;
    size_t size = n*n*sizeof(float);

    float A[16], B[16], C[16];

    for(int i=0;i<n*n;i++){
        A[i] = 1.0f;
        B[i] = 1.0f;
    }

    float *A_d, *B_d, *C_d;

    cudaMalloc((void**)&A_d, size);
    cudaMalloc((void**)&B_d, size);
    cudaMalloc((void**)&C_d, size);

    cudaMemcpy(A_d, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(B_d, B, size, cudaMemcpyHostToDevice);

    dim3 block_size(BLOCK_SIZE, BLOCK_SIZE);
    dim3 n_blocks(div_up(n, BLOCK_SIZE), div_up(n, BLOCK_SIZE));

    Multiplica_Matrices_GM<<<n_blocks, block_size>>>(C_d, A_d, B_d, n, n);

    cudaMemcpy(C, C_d, size, cudaMemcpyDeviceToHost);

    for(int i=0;i<n;i++){
        for(int j=0;j<n;j++){
            printf("%f ", C[i*n+j]);
        }
        printf("\n");
    }

    cudaFree(A_d);
    cudaFree(B_d);
    cudaFree(C_d);

    return 0;
}

Writing matmul.cu


In [ ]:
!nvcc matmul.cu -o matmul
!./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
4.000000 4.000000 4.000000 4.000000 
4.000000 4.000000 4.000000 4.000000 
4.000000 4.000000 4.000000 4.000000 
4.000000 4.000000 4.000000 4.000000 
